# Large-Scale Web Log Analysis Project

## Part 2: Silver Layer — Sessionization, Cleansing, Bot Filtering

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime, timedelta
import time

spark = SparkSession.builder \
    .appName('LogAnalysis-Part2-Silver') \
    .master('spark://spark-master:7077') \
    .config('spark.executor.memory', '1g') \
    .config('spark.sql.shuffle.partitions', '10') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.warehouse.dir', '/home/jovyan/data/warehouse') \
    .getOrCreate()

PROJECT = '/home/jovyan/data/log_analysis'
LAKE = f'{PROJECT}/lake'

bronze = spark.read.parquet(f'{LAKE}/bronze/clickstream')
print(f'Bronze: {bronze.count():,} rows')
print(f'✅ Spark UI: http://localhost:4040')

Bronze: 1,796,530건
✅ Spark UI: http://localhost:4040


---
## 1. Bot Traffic Detection and Filtering

In [2]:
# Bot detection rules
# 1. user_id starts with 'bot_' (known bots)
# 2. More than 50 events per session + average interval < 2 seconds
# 3. Browser is a known bot user agent (e.g. 'Python-urllib')

bot_browsers = ['Python-urllib', 'curl', 'wget', 'Scrapy', 'Googlebot']

# Rules 1 & 3: direct identification
known_bots = bronze.filter(
    F.col('user_id').startswith('bot_') |
    F.col('browser').isin(bot_browsers)
)

print(f'Known bot events: {known_bots.count():,}')
print(f'Known bot users: {known_bots.select("user_id").distinct().count():,}')

알려진 봇 이벤트: 921,647건
알려진 봇 유저: 6,785명


In [3]:
# Rule 2: behavior-based bot detection (session statistics)
session_stats = bronze.groupBy('session_id', 'user_id').agg(
    F.count('*').alias('event_count'),
    F.min('timestamp').alias('session_start'),
    F.max('timestamp').alias('session_end'),
).withColumn(
    'session_duration_sec',
    F.unix_timestamp('session_end') - F.unix_timestamp('session_start')
).withColumn(
    'avg_interval_sec',
    F.when(F.col('event_count') > 1,
           F.col('session_duration_sec') / (F.col('event_count') - 1))
    .otherwise(None)
)

# Suspicious sessions: 50+ events & avg interval < 2 seconds
suspicious_sessions = session_stats.filter(
    (F.col('event_count') > 50) & (F.col('avg_interval_sec') < 2)
)

suspicious_users = suspicious_sessions.select('user_id').distinct()
print(f'Behavior-based suspicious users: {suspicious_users.count():,}')

# Bot session stats vs regular session stats
print('\n=== Bot Sessions vs Regular Sessions ===')
session_stats.withColumn(
    'is_bot', F.col('user_id').startswith('bot_')
).groupBy('is_bot').agg(
    F.count('*').alias('sessions'),
    F.avg('event_count').alias('avg_events'),
    F.avg('avg_interval_sec').alias('avg_interval_sec'),
    F.avg('session_duration_sec').alias('avg_duration_sec'),
).show()

행동 기반 의심 유저: 6,739명

=== 봇 세션 vs 일반 세션 ===
+------+--------+------------------+------------------+-----------------+
|is_bot|sessions|        avg_events|  avg_interval_sec| avg_duration_sec|
+------+--------+------------------+------------------+-----------------+
| false|  139409| 6.275656521458442| 71.84077946516263|370.4623876507256|
|  true|    7317|125.95968293016263|1.0446929541107048|130.6655733223999|
+------+--------+------------------+------------------+-----------------+



In [4]:
# Remove bots
all_bot_users = known_bots.select('user_id').distinct() \
    .union(suspicious_users) \
    .distinct()

clean_df = bronze.join(all_bot_users, 'user_id', 'left_anti')

before = bronze.count()
after = clean_df.count()
print(f'Bot removal: {before:,} → {after:,} ({before - after:,} rows, {(before-after)/before*100:.1f}% removed)')

봇 제거: 1,796,530 → 874,883 (921,647건, 51.3% 제거)


---
## 2. Data Cleansing

In [5]:
# Cleansing rules
silver_df = clean_df \
    .filter(F.col('event_id').isNotNull()) \
    .filter(F.col('timestamp').isNotNull()) \
    .filter(F.col('status_code').isin(200, 301, 302)) \
    .dropDuplicates(['event_id']) \
    .withColumn('referrer', F.coalesce(F.col('referrer'), F.lit('direct'))) \
    .withColumn('is_mobile', F.col('device').isin('mobile', 'tablet')) \
    .drop('_ingested_at')

print(f'After cleansing: {silver_df.count():,} rows')
print(f'Removed error responses (4xx, 5xx) + duplicates + NULLs')

정제 후: 834,822건
에러 응답 제거 (4xx, 5xx) + 중복 제거 + NULL 제거


---
## 3. Sessionization

Sessions are split at 30 minutes of inactivity.

```
User events:  10:00  10:05  10:10  10:50  10:55  11:40
Gap:                  5min   5min   40min!  5min   45min!
Session:      ┌── Session1 ──┐  ┌─ Session2 ─┐  ┌Session3┐
```

In [6]:
SESSION_GAP_MINUTES = 30

# Sort by user + time, compute gap from previous event
w = Window.partitionBy('user_id').orderBy('timestamp')

sessionized = silver_df \
    .withColumn('prev_timestamp', F.lag('timestamp').over(w)) \
    .withColumn('gap_seconds',
        F.unix_timestamp('timestamp') - F.unix_timestamp('prev_timestamp')
    ) \
    .withColumn('new_session',
        F.when(
            F.col('prev_timestamp').isNull() |
            (F.col('gap_seconds') > SESSION_GAP_MINUTES * 60),
            F.lit(1)
        ).otherwise(F.lit(0))
    ) \
    .withColumn('computed_session_id',
        F.concat(
            F.col('user_id'), F.lit('_'),
            F.sum('new_session').over(w).cast('string')
        )
    ) \
    .drop('prev_timestamp', 'gap_seconds', 'new_session')

print(f'Sessionization complete')
print(f'Original session_id count: {silver_df.select("session_id").distinct().count():,}')
print(f'Computed session_id count: {sessionized.select("computed_session_id").distinct().count():,}')

sessionized.select('user_id', 'timestamp', 'event_type', 'session_id', 'computed_session_id') \
    .filter(F.col('user_id') == 'u_1') \
    .orderBy('timestamp') \
    .show(20, truncate=False)

세션화 완료
원본 session_id 수: 139,409
계산된 session_id 수: 137,319
+-------+--------------------------+------------+--------------+-------------------+
|user_id|timestamp                 |event_type  |session_id    |computed_session_id|
+-------+--------------------------+------------+--------------+-------------------+
|u_1    |2025-06-04 18:09:45.96009 |page_view   |s_1_1749060504|u_1_1              |
|u_1    |2025-06-04 18:10:16.493419|page_view   |s_1_1749060504|u_1_1              |
|u_1    |2025-06-04 18:11:46.504476|page_view   |s_1_1749060504|u_1_1              |
|u_1    |2025-06-04 18:12:46.645306|page_view   |s_1_1749060504|u_1_1              |
|u_1    |2025-06-04 18:14:14.386886|page_view   |s_1_1749060504|u_1_1              |
|u_1    |2025-06-04 18:15:29.070452|page_view   |s_1_1749060504|u_1_1              |
|u_1    |2025-06-04 18:16:19.550969|page_view   |s_1_1749060504|u_1_1              |
|u_1    |2025-06-07 22:56:52       |page_view   |s_1_1749337012|u_1_2              |
|u_1   

---
## 4. Silver Layer Save

In [7]:
# Silver events table
start = time.time()

silver_events = sessionized \
    .select(
        'event_id', 'user_id', 'computed_session_id',
        'event_type', 'page', 'product_id', 'timestamp',
        'device', 'browser', 'os', 'referrer', 'is_mobile',
        'response_time_ms', 'event_date', 'event_hour'
    ) \
    .withColumnRenamed('computed_session_id', 'session_id')

silver_events.write \
    .mode('overwrite') \
    .partitionBy('event_date') \
    .parquet(f'{LAKE}/silver/events')

elapsed = time.time() - start
print(f'Silver events: {silver_events.count():,} rows, {elapsed:.1f}s')

Silver 이벤트: 834,822건, 4.8초


In [8]:
# Silver sessions table (session-level aggregation)
start = time.time()

silver_sessions = silver_events.groupBy('session_id', 'user_id').agg(
    F.min('timestamp').alias('session_start'),
    F.max('timestamp').alias('session_end'),
    F.count('*').alias('event_count'),
    F.countDistinct('page').alias('unique_pages'),
    F.first('device').alias('device'),
    F.first('browser').alias('browser'),
    F.first('os').alias('os'),
    F.first('referrer').alias('referrer'),
    F.first('is_mobile').alias('is_mobile'),
    F.avg('response_time_ms').alias('avg_response_ms'),
    
    # Funnel event flags
    F.max(F.when(F.col('event_type') == 'product_view', 1).otherwise(0)).alias('has_product_view'),
    F.max(F.when(F.col('event_type') == 'add_to_cart', 1).otherwise(0)).alias('has_add_to_cart'),
    F.max(F.when(F.col('event_type') == 'checkout', 1).otherwise(0)).alias('has_checkout'),
    F.max(F.when(F.col('event_type') == 'purchase', 1).otherwise(0)).alias('has_purchase'),
).withColumn(
    'session_duration_sec',
    F.unix_timestamp('session_end') - F.unix_timestamp('session_start')
).withColumn(
    'session_date', F.to_date('session_start')
)

silver_sessions.write \
    .mode('overwrite') \
    .partitionBy('session_date') \
    .parquet(f'{LAKE}/silver/sessions')

elapsed = time.time() - start
print(f'Silver sessions: {silver_sessions.count():,} rows, {elapsed:.1f}s')

silver_sessions.show(5)

Silver 세션: 137,319건, 4.5초
+----------+-------+--------------------+--------------------+-----------+------------+-------+----------------+-------+--------+---------+------------------+----------------+---------------+------------+------------+--------------------+------------+
|session_id|user_id|       session_start|         session_end|event_count|unique_pages| device|         browser|     os|referrer|is_mobile|   avg_response_ms|has_product_view|has_add_to_cart|has_checkout|has_purchase|session_duration_sec|session_date|
+----------+-------+--------------------+--------------------+-----------+------------+-------+----------------+-------+--------+---------+------------------+----------------+---------------+------------+------------+--------------------+------------+
| u_10003_1|u_10003| 2025-06-01 21:28:54|2025-06-01 21:32:...|          5|           4| mobile|            Edge|  Linux|   naver|     true|             278.0|               0|              0|           0|           0| 

In [9]:
spark.stop()
print('Part 2 complete: Silver (sessionization, cleansing, bot filtering)')
print('Next: Part 3 - Gold (funnel analysis, anomaly detection, time patterns)')

Part 2 완료: Silver (세션화, 정제, 봇 필터링)
다음: Part 3 - Gold (퍼널 분석, 이상 탐지, 시간대 패턴)
